# Joint comparison of candidate tilt mechanisms

This final notebook is run after the component notebooks pass QC. It asks which mechanisms retain within-eddy and between-eddy associations after adjustment. Direction and magnitude are analysed separately, AE and CE are kept separate, and whole eddies remain the clustering unit.

The goal is not to declare causality from a single coefficient. A mechanism is considered supported only when its directional, magnitude, timescale, regime and within-eddy predictions agree.


In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

HERE = Path.cwd()
ANALYSIS_ROOT = HERE.parent
if str(ANALYSIS_ROOT) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_ROOT))
if str(ANALYSIS_ROOT / "beta_effect_background_flow") not in sys.path:
    sys.path.insert(0, str(ANALYSIS_ROOT / "beta_effect_background_flow"))

import seacofs_tilt_tools as tilt
import mechanism_tools as mech

paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
df, _ = tilt.load_tilt_tables(paths)
df = mech.require_tilt_measurements(df)
print(f"Rows: {len(df):,}; measured tilts: {df.TiltDis.notna().sum():,}; eddies: {df.Eddy.nunique():,}")


Rows: 127,426; measured tilts: 105,621; eddies: 2,982


In [2]:
from beta_effect_background_flow.background_flow_tools import BackgroundConfig, load_background_cache

N2_CACHE = Path("/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset/tilt_mechanisms/n2_eddy_day_v3_core.parquet")
background = load_background_cache(BackgroundConfig()).drop(columns=["ic", "jc", "month"], errors="ignore")
n2 = pd.read_parquet(N2_CACHE)

data = mech.merge_one_to_one_or_many_to_one(df, background)
data = mech.merge_one_to_one_or_many_to_one(data, n2)
for depth in (200, 500):
    data[f"N2_{depth}m_s2"] = data[f"N2_{depth}m_core_s2"]
data = tilt.add_pv_gradient_terms(data, grid)
data = mech.add_stratification_proxies(data)
data = mech.add_background_shear(data)
data = mech.add_accumulated_shear(data, "ann_500", windows=(10, 20, 30))
data = tilt.add_region_labels(data, grid)
data = mech.add_topographic_regimes(data, shelf_depth=2000.0, dominance_ratio=2.0)
data["slope_mag"] = np.hypot(data.dhdx, data.dhdy)


In [3]:
# Predeclared mechanism variables. Add wind only after its cache passes QC.
mechanisms = [
    "beta", "N2_500m_s2", "N2_over_f2_500m", "Bu_proxy_500m",
    "ann_500_shear_mag_ms", "ann_500_accum_20d_mag_km",
    "slope_mag", "topo_plan_ratio_raw", "Rc", "h",
]
varying = ["ann_500_shear_mag_ms", "ann_500_accum_20d_mag_km", "Rc"]
data = mech.standardise_within_between(data, varying)
display(data[mechanisms].describe().T)


,count,mean,std,min,25%,50%,75%,max
beta,127424.0,-1.917380e-11,8.117010e-13,-2.082975e-11,-1.991064e-11,-1.908482e-11,-1.846449e-11,-1.755648e-11
N2_500m_s2,127424.0,7.513177e-05,1.578091e-05,3.847153e-05,6.349761e-05,7.335587e-05,8.545332e-05,3.362520e-04
N2_over_f2_500m,127424.0,1.195464e+04,4.418928e+03,5.253308e+03,8.593612e+03,1.102799e+04,1.454184e+04,7.474683e+04
Bu_proxy_500m,127424.0,8.312354e-02,9.202036e-02,3.101624e-03,2.933069e-02,5.490975e-02,1.023250e-01,3.577808e+00
ann_500_shear_mag_ms,127426.0,7.296612e-02,4.326004e-02,0.000000e+00,4.061567e-02,6.571631e-02,9.770739e-02,5.380698e-01
ann_500_accum_20d_mag_km,124444.0,5.363503e+01,3.675332e+01,8.788898e-02,2.541893e+01,4.751640e+01,7.439313e+01,4.291230e+02
slope_mag,127424.0,1.402150e-02,2.227736e-02,1.098551e-05,2.035181e-03,4.430007e-03,1.540749e-02,1.637708e-01
topo_plan_ratio_raw,127424.0,2.644932e+01,7.865258e+01,5.759771e-03,1.725192e+00,4.375227e+00,1.810903e+01,1.491268e+03
Rc,127426.0,7.862359e+01,3.444922e+01,1.484732e+01,5.290895e+01,7.246890e+01,9.816889e+01,2.845415e+02
h,127426.0,4.082367e+03,1.042393e+03,6.510651e+01,3.654601e+03,4.606874e+03,4.750918e+03,4.942000e+03


In [4]:
# Magnitude: compare standardized clustered models, then inspect effect sizes and CIs.
import statsmodels.formula.api as smf

model_sets = {
    "environment": ["beta", "N2_over_f2_500m", "Rc", "h"],
    "plus_shear": ["beta", "N2_over_f2_500m", "Rc", "h",
                   "ann_500_accum_20d_mag_km_between", "ann_500_accum_20d_mag_km_within"],
    "plus_topography": ["beta", "N2_over_f2_500m", "Rc", "h", "slope_mag", "topo_plan_ratio_raw",
                        "ann_500_accum_20d_mag_km_between", "ann_500_accum_20d_mag_km_within"],
}
fits = {}
for cyc, part in data.groupby("Cyc"):
    for name, columns in model_sets.items():
        use = part[["Eddy", "TiltDis", *columns]].replace([np.inf, -np.inf], np.nan).dropna().copy()
        for column in columns:
            sd = use[column].std()
            use[f"z_{column}"] = (use[column] - use[column].mean()) / sd if sd > 0 else 0
        formula = "np.log1p(TiltDis) ~ " + " + ".join(f"z_{c}" for c in columns)
        fits[(cyc, name)] = smf.gee(formula, groups="Eddy", data=use).fit()
        print(cyc, name, "rows", len(use), "eddies", use.Eddy.nunique())


AE environment rows 53607 eddies 1422
AE plus_shear rows 53607 eddies 1422
AE plus_topography rows 53607 eddies 1422
CE environment rows 52014 eddies 1531
CE plus_shear rows 52014 eddies 1531
CE plus_topography rows 52014 eddies 1531


In [5]:
for key, fit in fits.items():
    print("\n", key)
    display(pd.DataFrame({"estimate": fit.params, "ci_low": fit.conf_int()[0], "ci_high": fit.conf_int()[1]}))



 ('AE', 'environment')


,estimate,ci_low,ci_high
Intercept,2.986349,2.954461,3.018237
z_beta,-0.115270,-0.168450,-0.062090
z_N2_over_f2_500m,0.191753,0.136989,0.246517
z_Rc,-0.114679,-0.138705,-0.090653
z_h,-0.017608,-0.041973,0.006756



 ('AE', 'plus_shear')


,estimate,ci_low,ci_high
Intercept,2.986349,2.955097,3.017600
z_beta,-0.104726,-0.158153,-0.051300
z_N2_over_f2_500m,0.202465,0.148013,0.256918
z_Rc,-0.120734,-0.144238,-0.097230
z_h,-0.010686,-0.035220,0.013848
z_ann_500_accum_20d_mag_km_between,-0.063626,-0.090832,-0.036421
z_ann_500_accum_20d_mag_km_within,-0.018519,-0.035193,-0.001846



 ('AE', 'plus_topography')


...

,estimate,ci_low,ci_high
Intercept,2.812133,2.785130,2.839136
z_beta,-0.353005,-0.397449,-0.308560
z_N2_over_f2_500m,-0.082479,-0.124228,-0.040730
z_Rc,-0.063907,-0.086807,-0.041007
z_h,-0.027448,-0.052391,-0.002505



 ('CE', 'plus_shear')


,estimate,ci_low,ci_high
Intercept,2.812133,2.785948,2.838318
z_beta,-0.308341,-0.353200,-0.263481
z_N2_over_f2_500m,-0.031235,-0.073909,0.011438
z_Rc,-0.081687,-0.104296,-0.059077
z_h,-0.017812,-0.042140,0.006515
z_ann_500_accum_20d_mag_km_between,-0.113796,-0.138321,-0.089270
z_ann_500_accum_20d_mag_km_within,0.003869,-0.011523,0.019261



 ('CE', 'plus_topography')


,estimate,ci_low,ci_high
Intercept,2.812133,2.786254,2.838012
z_beta,-0.298463,-0.343785,-0.253142
z_N2_over_f2_500m,-0.022832,-0.065509,0.019844
z_Rc,-0.090033,-0.112771,-0.067295
z_h,-0.064855,-0.094261,-0.035448
z_slope_mag,-0.052171,-0.077818,-0.026523
z_topo_plan_ratio_raw,-0.038461,-0.070016,-0.006906
z_ann_500_accum_20d_mag_km_between,-0.103598,-0.128196,-0.079000
z_ann_500_accum_20d_mag_km_within,0.001634,-0.013833,0.017102


In [6]:
# Directional scorecard assembled from the component notebooks.
direction_metrics = [
    "ann_500_tilt_shear_offset",
    "ann_500_accum_20d_offset",
]
data["tilt_planetary_pv_offset"] = mech.signed_angle_difference(data.TiltDir, data.PV_grad_plan_theta)
data["tilt_topographic_pv_offset"] = mech.signed_angle_difference(data.TiltDir, data.PV_grad_topo_theta)
direction_metrics += ["tilt_planetary_pv_offset", "tilt_topographic_pv_offset"]
scorecard = mech.circular_offset_summary(data, direction_metrics, group=("Cyc", "ShelfRegime", "PVRegime"))
display(scorecard.sort_values(["Cyc", "ShelfRegime", "resultant_length"], ascending=[True, True, False]))


,Cyc,ShelfRegime,PVRegime,metric,eddies,mean_offset_deg,resultant_length,median_abs_offset_deg
10,AE,off_shelf,topographic,tilt_planetary_pv_offset,1224,-161.856367,0.329452,192.569651
2,AE,off_shelf,mixed,tilt_planetary_pv_offset,996,-172.497365,0.280772,184.720250
6,AE,off_shelf,planetary,tilt_planetary_pv_offset,524,-175.561068,0.238133,183.091162
1,AE,off_shelf,mixed,ann_500_accum_20d_offset,996,-105.398316,0.158796,207.180754
0,AE,off_shelf,mixed,ann_500_tilt_shear_offset,996,-121.279669,0.145468,207.396679
5,AE,off_shelf,planetary,ann_500_accum_20d_offset,524,-119.172209,0.133716,201.158059
9,AE,off_shelf,topographic,ann_500_accum_20d_offset,1224,-83.626268,0.129103,210.158194
4,AE,off_shelf,planetary,ann_500_tilt_shear_offset,524,-148.505811,0.102699,190.433139
8,AE,off_shelf,topographic,ann_500_tilt_shear_offset,1224,-90.836460,0.090642,204.383417
11,AE,off_shelf,topographic,tilt_topographic_pv_offset,1224,-62.110776,0.060111,193.112874


## Final evidence rubric

For each mechanism report:

1. **Direction:** Is the eddy-equal offset concentrated around the predicted direction?
2. **Magnitude:** Does forcing magnitude predict `TiltDis` with a scientifically meaningful effect?
3. **Timescale:** Does a plausible trailing window outperform instantaneous forcing?
4. **Within eddies:** Does the same eddy respond when forcing changes?
5. **Regime:** Does the relationship strengthen where the mechanism should dominate?
6. **Robustness:** Does it survive spatial blocks, tilt thresholds, depth choice and background definition?

Use language such as “supports,” “is consistent with,” or “does not support.” Reserve causal language for a coherent suite of predictions, not isolated significance.
